# 1. Carga de datos

Este notebook carga el dataset de entrada, valida su estructura y persiste los datos en formato Delta para que puedan ser reutilizados por los notebooks posteriores.

In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("control-2-carga")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.conf.set("spark.sql.adaptive.enabled", "true")

raw_path = Path("data/raw")
raw_path.mkdir(parents=True, exist_ok=True)
print(f"Directorio de entrada: {raw_path.resolve()}")

In [ ]:
import os
import pandas as pd

csv_files = list(Path("data/raw").glob("*.csv"))

if csv_files:
    source_file = csv_files[0]
    print(f"Cargando dataset desde: {source_file}")
    raw_df = spark.read.option("header", True).option("inferSchema", True).csv(str(source_file))
else:
    print("No se encontró un CSV en data/raw. Se generará un dataset sintético para continuar.")
    synthetic_df = pd.DataFrame(
        {
            "feature_1": [0.1, 0.5, 1.2, 2.1, 3.0, 4.2, 5.1, 6.3, 7.2, 8.1],
            "feature_2": [1.2, 2.5, 1.8, 3.1, 4.0, 5.3, 2.4, 6.1, 7.0, 8.2],
            "feature_3": [5, 6, 7, 8, 9, 10, 11, 12, 13, 14],
            "target": [0, 0, 1, 0, 1, 1, 1, 0, 1, 0],
        }
    )
    source_file = Path("data/raw") / "synthetic_dataset.csv"
    synthetic_df.to_csv(source_file, index=False)
    raw_df = spark.read.option("header", True).option("inferSchema", True).csv(str(source_file))

raw_df = raw_df.withColumn("ingested_at", F.current_timestamp())
raw_df.printSchema()
raw_df.show(5, truncate=False)

raw_df.write.mode("overwrite").format("delta").saveAsTable("bronze_control_2")
print("Datos cargados y persistidos en Delta como bronze_control_2")